# COMP34812 Coursework - Cross Encoder

## Install dependency

In [ ]:
from pathlib import Path
root_path = Path().resolve().parent.parent
print(f"root_path: {root_path}")

# For non-Kaggle working environment
# !pip install -r {root_path}/requirements.txt
# For Kaggle working environment
!pip install matplotlib pandas sentence_transformers torch transformers

print(f"pip install is finished ...")

## Import dependency

In [ ]:
import matplotlib.pyplot as plt
import os
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

print(f"Import is finished ...")

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset


class AVDataset(Dataset):
    def __init__(self, csv_file_normal: str, csv_file_manual_stylometry: str, text_1_column="text_1", text_2_column="text_2", author_score_column="author_score", style_similarity_score_column="style_similarity_score", separation_token="[SEP]"):
        # Load CSV
        self.df_normal = pd.read_csv(csv_file_normal)
        self.df_manual_stylomtry = pd.read_csv(csv_file_manual_stylometry)

        assert len(self.df_normal) == len(self.df_manual_stylomtry), f"Mismatch between lengths of dataframes: {len(self.df_normal)} != {len(self.df_manual_stylomtry)}"

        # Columns
        self.text_1 = self.df_normal[text_1_column].astype(str).tolist()
        self.text_2 = self.df_normal[text_2_column].astype(str).tolist()
        self.author_score = self.df_normal[author_score_column].astype(float).tolist()
        self.style_similarity_score = self.df_normal[style_similarity_score_column].astype(float).tolist()

        # Load the whole csv file
        self.manual_stylomtries = self.df_manual_stylomtry.values

        assert len(self.df_normal) == len(self.manual_stylomtries), f"Mismatch between lengths between self.df_normal and self.manual_stylometries: {len(self.df_normal)} != {len(self.manual_stylomtries)}"

        self.separation_token = separation_token

        # Sanity check
        assert len(self.text_1) == len(self.author_score) and len(self.text_2) == len(self.author_score) and len(self.style_similarity_score) == len(
            self.author_score), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)}, author_score={len(self.author_score)}, style_similarity_score={len(self.style_similarity_score)}"

    def __len__(self):
        return len(self.author_score)

    def __getitem__(self, idx):
        """
        batch_manual_stylometry_y is literally author_score, which is the label
        """
        # Return a tuple
        combined_text = (self.text_1[idx], self.text_2[idx])
        author_score = torch.tensor(self.author_score[idx], dtype=torch.float)
        style_similarity_score = torch.tensor(
            self.style_similarity_score[idx], dtype=torch.float)
        manual_stylometry = torch.tensor(self.manual_stylomtries[idx], dtype=torch.float)        

        return (combined_text, author_score, style_similarity_score, manual_stylometry, author_score)


In [ ]:
import torch
from torch import nn, functional as F
from transformers import AutoModel

"""
Return different scores, which will be used to determine the final score in the task of Author Verification
"""


class MultiHeadCrossEncoder(nn.Module):
    model_name: str
    head_weights: torch.Tensor

    def __init__(self, model_name: str, head_weights: torch.Tensor, manual_stylometry_dim: int = 171):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        encoder_hidden_size = self.encoder.config.hidden_size

        self.shared = nn.Linear(encoder_hidden_size, encoder_hidden_size)
        self.shared_activation = nn.ReLU()

        self.head_author = nn.Linear(encoder_hidden_size, 1)
        self.head_style_similarity = nn.Linear(encoder_hidden_size, 1)
        self.head_manual_stylometry = nn.Sequential(
            nn.Linear(manual_stylometry_dim, 64), 
            nn.ReLU(), 
            nn.Linear(64, 1)
        )

        assert isinstance(
            head_weights, torch.Tensor), f"Invalid type of head_weights: {type(head_weights)}"
        assert head_weights.dim() == 1 and head_weights.numel() == 3, f"Invalid shape of head_weights (expect [3]): {head_weights.shape}"

        if sum(head_weights) == 1:
            self.head_weights = head_weights
        else:
            # Apply softmax function
            self.head_weights = F.softmax(head_weights, dim=0)

    def forward(self, input_ids, attention_mask, manual_stylometry_vector: torch.Tensor, token_type_ids=None):
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        embedding_cls = encoder_outputs.last_hidden_state[:, 0]

        shared_output = self.shared_activation(self.shared(embedding_cls))

        # Individual heads
        score_author = self.head_author(shared_output)
        score_style_similarity = self.head_style_similarity(shared_output)
        score_manual_stylometry = self.head_manual_stylometry(manual_stylometry_vector)

        score_final = (self.head_weights[0] * score_author) + (self.head_weights[1] * score_style_similarity) + (self.head_weights[2] * score_manual_stylometry)

        return {
            f"score_final": score_final,
            f"score_author": score_author,
            f"score_style_similarity": score_style_similarity, 
            f"score_manual_stylometry": score_manual_stylometry
        }


## Global variable

In [ ]:
HEAD_WEIGHTS = torch.tensor([0.3, 0.3, 0.4], dtype=torch.float)
# The name of the model that both tokenizer and the embedder use (must use the same)
# Consider MODEL_NAME as a specific language used by both tokenizer and the embedder for communication, i.e. they know which token_id represents which token
MODEL_NAME = f"bert-base-uncased"
SEPARATION_TOKEN = f"[SEP]"
# Larger batch for efficiency
BATCH_SIZE = 32
SHUFFLE_DATA = False
TRAIN_EPOCH = 5
LEARNING_RATE = 2e-5
# Number of train_epoch to log once
EPOCH_PER_LOG = 1

ROOT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))

KAGGLE_DATASET_NAME = f"authorverification4"

# TODO: Update me when applicable
# CSV_NAME_TRAIN_NORMAL = f"train_final.csv"
CSV_NAME_TRAIN_NORMAL = f"AV_trial_final.csv"
# TODO: Update me when applicable
# For non-Kaggle working environment
# CSV_PATH_TRAIN_NORMAL = os.path.join(ROOT_PATH, "data", "AV", CSV_NAME_TRAIN_NORMAL)
# For Kaggle working environment
CSV_PATH_TRAIN_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TRAIN_NORMAL)

CSV_NAME_VALIDATION_NORMAL = f"dev_final.csv"
# TODO: Update me when applicable
# For non-Kaggle working environment
# CSV_PATH_VALIDATION_NORMAL = os.path.join(ROOT_PATH, "data", "AV", CSV_NAME_VALIDATION_NORMAL)
# For Kaggle working environment
CSV_PATH_VALIDATION_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_VALIDATION_NORMAL)

# TODO: Update me when applicable
# CSV_NAME_TRAIN_MANUAL_STYLOMETRY = f"train_style_diff.csv"
CSV_NAME_TRAIN_MANUAL_STYLOMETRY = f"trial_style_diff.csv"
# For Kaggle working environment
CSV_PATH_TRAIN_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TRAIN_MANUAL_STYLOMETRY)

CSV_NAME_VALIDATION_MANUAL_STYLOMETRY = f"val_style_diff.csv"
# For Kaggle working environment
CSV_PATH_VALIDATION_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_VALIDATION_MANUAL_STYLOMETRY)

CSV_NAME_OUTPUT = f"predictions.csv"
# For Kaggle working environment
CSV_PATH_OUTPUT = os.path.join(ROOT_PATH, "kaggle", "working", CSV_NAME_OUTPUT)

CROSS_ENCODER_NAME = f"cross-encoder.pth"
# TODO: Update me when applicable
# For non-Kaggle working environment
# CROSS_ENCODER_PATH = os.path.join(ROOT_PATH, "data", "model", CROSS_ENCODER_NAME)
# For Kaggle working environment
CROSS_ENCODER_PATH = os.path.join(ROOT_PATH, "kaggle", "working", CROSS_ENCODER_NAME)

print(f"Global variables are set ...")

## Utils

In [ ]:
def log(text=None) -> None:
    if text is None:
        print(f"[INFO]")
    else:
        print(f"[INFO] {text}")

## Prepare data

In [ ]:
log(f"Reading csv file at {CSV_PATH_TRAIN_NORMAL} ...")
    
# Load dataset
# Training
dataset_train = AVDataset(CSV_PATH_TRAIN_NORMAL, CSV_PATH_TRAIN_MANUAL_STYLOMETRY)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)
# Validation
dataset_validation = AVDataset(CSV_PATH_VALIDATION_NORMAL, CSV_PATH_VALIDATION_MANUAL_STYLOMETRY)
dataloader_validation = DataLoader(dataset_validation, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)
# Testing
# TODO: Update csv path
dataset_test = AVDataset(CSV_PATH_VALIDATION_NORMAL, CSV_PATH_VALIDATION_MANUAL_STYLOMETRY)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)

print(f"All data finish preparing ...")

## Instantiate entity

In [ ]:
device = torch.device(f"cuda" if torch.cuda.is_available() else f"cpu")
log(f"Using device {device} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
cross_encoder = MultiHeadCrossEncoder(
    model_name=MODEL_NAME, 
    head_weights=HEAD_WEIGHTS
).to(device)
optimizer = AdamW(cross_encoder.parameters(), lr=LEARNING_RATE)
# This expects raw numbers (logits) and handles the Sigmoid internally
criterion = nn.BCEWithLogitsLoss()

print(f"All instances have been instantiated ...")

## Train model
- Note that this will also calculate the testing loss of the model

In [ ]:
losses_train = {
    f"loss": [], 
    f"loss_author": [], 
    f"loss_style_similarity": [],
    f"loss_manual_stylometry": []
}
losses_validation = {
    f"loss": [], 
    f"loss_author": [], 
    f"loss_style_similarity": [],
    f"loss_manual_stylometry": [], 
    f"f1": []
}


for epoch in range(TRAIN_EPOCH):
    # =============================== Training ===============================
    metrics_validation = {
        # True positive
        f"tp": 0, 
        # False positive
        f"fp": 0, 
        # True negative
        f"tn": 0, 
        # False negative
        f"fn": 0
    }

    cross_encoder.train()

    average_loss_train = 0.0
    average_loss_train_author = 0.0
    average_loss_train_style_similarity = 0.0
    average_loss_train_manual_stylometry = 0.0

    for batch_X, batch_author_score, batch_style_similarity_score, batch_manual_stylometry_X, batch_manual_stylometry_y in dataloader_train:
        batch_author_score = batch_author_score.to(device)
        batch_style_similarity_score = batch_style_similarity_score.to(device)
        batch_manual_stylometry_X = batch_manual_stylometry_X.to(device)
        batch_manual_stylometry_y = batch_manual_stylometry_y.to(device)
    
        # Reset optimizer for the current gradient descent
        optimizer.zero_grad()

        """
        {
            "input_ids": tensor, 
            "token_type_ids": tensor, 
            "attention_mask": tensor
        }
        """
        encoding_X = tokenizer(
            batch_X[0], 
            batch_X[1], 
            padding=True, 
            truncation=True, 
            return_tensors=f"pt"
        ).to(device)

        y_predictions = cross_encoder(
            input_ids=encoding_X["input_ids"], 
            attention_mask=encoding_X["attention_mask"],
            token_type_ids=encoding_X.get("token_type_ids"),
            manual_stylometry_vector=batch_manual_stylometry_X
        )

        # Calculate individual losses
        loss_train_author = criterion(y_predictions["score_author"].view(-1), batch_author_score)
        loss_train_style_similarity = criterion(y_predictions["score_style_similarity"].view(-1), batch_style_similarity_score)
        loss_train_manual_stylomtry = criterion(y_predictions["score_manual_stylometry"].view(-1), batch_manual_stylometry_y)

        # Combine loss
        loss_train = HEAD_WEIGHTS[0] * loss_train_author + HEAD_WEIGHTS[1] * loss_train_style_similarity+  HEAD_WEIGHTS[2] * loss_train_manual_stylomtry

        average_loss_train += loss_train.item()
        average_loss_train_author += loss_train_author.item()
        average_loss_train_style_similarity += loss_train_style_similarity.item()
        average_loss_train_manual_stylometry += loss_train_manual_stylomtry.item()

        # Apply gradient descent
        loss_train.backward()
        optimizer.step()


    # Update data to be plotted on graphs
    losses_train[f"loss"].append(average_loss_train / len(dataloader_train))
    losses_train[f"loss_author"].append(average_loss_train_author / len(dataloader_train))
    losses_train[f"loss_style_similarity"].append(average_loss_train_style_similarity / len(dataloader_train))
    losses_train[f"loss_manual_stylometry"].append(average_loss_train_manual_stylometry / len(dataloader_train))
    
    if epoch % EPOCH_PER_LOG == 0:
        # Log
        log(f"[EPOCH {epoch + 1} / {TRAIN_EPOCH}] Training loss: {losses_train[f'loss'][-1]}")
    
    # =============================== Testing ===============================
    cross_encoder.eval()

    average_loss_validation = 0.0
    average_loss_validation_author = 0.0
    average_loss_validation_style_similarity = 0.0
    average_loss_validation_manual_stylometry = 0.0

    with torch.no_grad():
        for batch_X, batch_author_score, batch_style_similarity_score, batch_manual_stylometry_X, batch_manual_stylometry_y in dataloader_validation:
            batch_author_score = batch_author_score.to(device)
            batch_style_similarity_score = batch_style_similarity_score.to(device)
            batch_manual_stylometry_X = batch_manual_stylometry_X.to(device)
            batch_manual_stylometry_y = batch_manual_stylometry_y.to(device)

            """
            {
                "input_ids": tensor, 
                "token_type_ids": tensor, 
                "attention_mask": tensor
            }
            """
            encoding_X = tokenizer(
                batch_X[0], 
                batch_X[1], 
                padding=True, 
                truncation=True, 
                return_tensors=f"pt"
            ).to(device)

            y_predictions = cross_encoder(
                input_ids=encoding_X["input_ids"], 
                attention_mask=encoding_X["attention_mask"],
                token_type_ids=encoding_X.get("token_type_ids"),
                manual_stylometry_vector=batch_manual_stylometry_X
            )

            # Calculate individual losses
            loss_validation_author = criterion(y_predictions["score_author"].detach().view(-1), batch_author_score)
            loss_validation_style_similarity = criterion(y_predictions["score_style_similarity"].detach().view(-1), batch_style_similarity_score)
            loss_validation_manual_stylomtry = criterion(y_predictions["score_manual_stylometry"].detach().view(-1), batch_manual_stylometry_y)

            # Combine loss
            loss_validation = HEAD_WEIGHTS[0] * loss_validation_author + HEAD_WEIGHTS[1] * loss_validation_style_similarity + HEAD_WEIGHTS[2] * loss_validation_manual_stylomtry

            average_loss_validation += loss_validation.item()
            average_loss_validation_author += loss_validation_author.item()
            average_loss_validation_style_similarity += loss_validation_style_similarity.item()
            average_loss_validation_manual_stylometry += loss_validation_manual_stylomtry.item()

            # Update metrics
            # Shape: [batch_size]
            y_prediction_final = torch.sigmoid(y_predictions["score_final"].detach().view(-1))

            labels_predicted = y_prediction_final.tolist()
            labels_actual = batch_author_score.tolist()

            assert len(labels_predicted) == len(labels_actual), f"Mismatch between lengths of labels_predicted and labels_actual: {len(labels_predicted)} != {len(labels_actual)}"

            for i in range(len(labels_predicted)):
                label_predicted = labels_predicted[i]
                label_actual = labels_actual[i]

                if label_actual == 1:
                    # True positive/false negative
                    if label_predicted >= 0.5:
                        # True positive
                        metrics_validation["tp"] += 1
                    else:
                        # False negative
                        metrics_validation["fn"] += 1
                elif label_actual == 0:
                    # True negative/false positive
                    if label_predicted >= 0.5:
                        # False positive
                        metrics_validation["fp"] += 1
                    else:
                        # True negative
                        metrics_validation["tn"] += 1
                else:
                    raise Exception(f"Invalid lable_actual (expect 0 or 1): {label_actual}")

        # Update data to be plotted on graphs
        losses_validation[f"loss"].append(average_loss_validation / len(dataloader_validation))
        losses_validation[f"loss_author"].append(average_loss_validation_author / len(dataloader_validation))
        losses_validation[f"loss_style_similarity"].append(average_loss_validation_style_similarity / len(dataloader_validation))
        losses_validation[f"loss_manual_stylometry"].append(average_loss_validation_manual_stylometry / len(dataloader_validation))

        # Calculate F1 score
        precision = metrics_validation[f"tp"] / (metrics_validation[f"tp"] + metrics_validation[f"fp"])
        recall = metrics_validation[f"tp"] / (metrics_validation[f"tp"] + metrics_validation[f"fn"])
        f1_score = (2 * precision * recall) / (precision + recall)

        losses_validation[f"f1"].append(f1_score)
    
    if epoch % EPOCH_PER_LOG == 0:
        # Log
        log(f"[EPOCH {epoch + 1} / {TRAIN_EPOCH}] Testing loss: {losses_validation[f'loss'][-1]}")

## Plot graph

### Loss

In [ ]:
NAME_X = f"Epoch"
NAME_Y = f"Loss"
epochs = range(1, TRAIN_EPOCH + 1)

plt.figure(figsize=(8,5))

plt.plot(epochs, losses_train[f"loss"], label=f"Training")
plt.plot(epochs, losses_validation[f"loss"], label=f"Testing")

plt.xlabel(NAME_X)
plt.ylabel(NAME_Y)
plt.title(f"{NAME_Y} (Training vs Testing)")
plt.legend()
plt.grid(True)

plt.show()

### Author Score Loss

In [ ]:
NAME_X = f"Epoch"
NAME_Y = f"Author Score Loss"
epochs = range(1, TRAIN_EPOCH + 1)

plt.figure(figsize=(8,5))

plt.plot(epochs, losses_train[f"loss_author"], label=f"Training")
plt.plot(epochs, losses_validation[f"loss_author"], label=f"Testing")

plt.xlabel(NAME_X)
plt.ylabel(NAME_Y)
plt.title(f"{NAME_Y} (Training vs Testing)")
plt.legend()
plt.grid(True)

plt.show()

### Style Similarity Score Loss

In [ ]:
NAME_X = f"Epoch"
NAME_Y = f"Style Similarity Score Loss"
epochs = range(1, TRAIN_EPOCH + 1)

plt.figure(figsize=(8,5))

plt.plot(epochs, losses_train[f"loss_style_similarity"], label=f"Training")
plt.plot(epochs, losses_validation[f"loss_style_similarity"], label=f"Testing")

plt.xlabel(NAME_X)
plt.ylabel(NAME_Y)
plt.title(f"{NAME_Y} (Training vs Testing)")
plt.legend()
plt.grid(True)

plt.show()

### Manual Stylometry Loss

In [ ]:
NAME_X = f"Epoch"
NAME_Y = f"Manual Stylometry Loss"
epochs = range(1, TRAIN_EPOCH + 1)

plt.figure(figsize=(8,5))

plt.plot(epochs, losses_train[f"loss_manual_stylometry"], label=f"Training")
plt.plot(epochs, losses_validation[f"loss_manual_stylometry"], label=f"Testing")

plt.xlabel(NAME_X)
plt.ylabel(NAME_Y)
plt.title(f"{NAME_Y} (Training vs Testing)")
plt.legend()
plt.grid(True)

plt.show()

### F1 Score

In [ ]:
NAME_X = f"Epoch"
NAME_Y = f"F1 Score"
epochs = range(1, TRAIN_EPOCH + 1)

plt.figure(figsize=(8,5))

plt.plot(epochs, losses_validation[f"f1"], label=f"Testing")

plt.xlabel(NAME_X)
plt.ylabel(NAME_Y)
plt.title(f"{NAME_Y}")
plt.legend()
plt.grid(True)

plt.show()

## Save trained model

In [ ]:
# Save model weights
torch.save(cross_encoder.state_dict(), CROSS_ENCODER_PATH)

log(f"Model weights are saved to {CROSS_ENCODER_PATH} ...")

## Run model in inference mode

In [ ]:
cross_encoder = MultiHeadCrossEncoder(model_name=MODEL_NAME, head_weights=HEAD_WEIGHTS).to(device)
cross_encoder.load_state_dict(torch.load(CROSS_ENCODER_PATH))

log(f"Cross encoder is loaded from {CROSS_ENCODER_PATH} ...")

cross_encoder.eval()

total_loss_test = 0.0
total_loss_author_test = 0.0
total_loss_style_similarity_test = 0.0
total_loss_manual_stylometry_test = 0.0

# Will be saved in predictions.csv
predictions = []

with torch.no_grad():
    for batch_X, batch_author_score, batch_style_similarity_score, batch_manual_stylometry_X, batch_manual_stylometry_y in dataloader_test:
        batch_author_score = batch_author_score.to(device)
        batch_style_similarity_score = batch_style_similarity_score.to(device)
        batch_manual_stylometry_X = batch_manual_stylometry_X.to(device)
        batch_manual_stylometry_y = batch_manual_stylometry_y.to(device)

        """
        {
            "input_ids": tensor, 
            "token_type_ids": tensor, 
            "attention_mask": tensor
        }
        """
        encoding_X = tokenizer(
            batch_X[0], 
            batch_X[1], 
            padding=True, 
            truncation=True, 
            return_tensors=f"pt"
        ).to(device)

        y_predictions = cross_encoder(
            input_ids=encoding_X["input_ids"], 
            attention_mask=encoding_X["attention_mask"],
            token_type_ids=encoding_X.get("token_type_ids"),
            manual_stylometry_vector=batch_manual_stylometry_X
        )

        # Shape: [batch_size]
        y_prediction_final = torch.sigmoid(y_predictions["score_final"].detach().view(-1))

        predictions.append(y_prediction_final.tolist())

        # Calculate individual losses
        loss_test_author = criterion(y_predictions["score_author"].detach().view(-1), batch_author_score)
        loss_test_style_similarity = criterion(y_predictions["score_style_similarity"].detach().view(-1), batch_style_similarity_score)
        loss_test_manual_stylometry = criterion(y_predictions["score_manual_stylometry"].detach().view(-1), batch_manual_stylometry_y)

        # Combine loss
        loss_test = HEAD_WEIGHTS[0] * loss_test_author + HEAD_WEIGHTS[1] * loss_test_style_similarity + HEAD_WEIGHTS[2] * loss_test_manual_stylometry

        total_loss_test += loss_test.item()
        total_loss_author_test += loss_test_author.item()
        total_loss_style_similarity_test += loss_test_style_similarity.item()
        total_loss_manual_styloemtry_test += loss_test_manual_stylometry.item()

# Calculate average testing loss
average_loss_test = total_loss_test / len(dataloader_test)
average_loss_author_test = total_loss_author_test / len(dataloader_test)
average_loss_style_similarity_test = total_loss_style_similarity_test / len(dataloader_test)
average_loss_manual_stylometry_test = total_loss_manual_stylometry_test / len(dataloader_test)

log(f"Testing loss: {average_loss_test:.4f}")
log(f"Testing author loss: {average_loss_author_test:.4f}")
log(f"Testing style similarity loss: {average_loss_style_similarity_test:.4f}")
log(f"Testing manual stylometry loss: {average_loss_manual_stylometry_test:.4f}")

### Generate Predictions CSV File

In [ ]:
df_predictions = pd.DataFrame(predictions, index=False)

df_predictions.to_csv(CSV_PATH_OUTPUT)

print(f"Predictions are saved to {CSV_PATH_OUTPUT} ...")